In [3]:
import itertools, time
import pulp
from jsp import load

In [26]:

CPX = "/opt/ibm/ILOG/CPLEX_Academic/cplex/bin/x86-64_linux/cplex"


In [20]:
def build(instance):
    prob = pulp.LpProblem("jsp", pulp.LpMinimize)

    s = {}
    for j, job in enumerate(instance):
        for k in range(len(job)):
            s[(j,k)] = pulp.LpVariable(f"s_{j}_{k}", lowBound=0)

    Cmax = pulp.LpVariable("Cmax", lowBound=0)
    prob += Cmax

    for j, job in enumerate(instance):
        for k in range(len(job) -1 ):
            prob += s[(j,k+1)] >= s[(j,k)] + job[k][1]

        prob += Cmax >= s[(j, len(job) -1)] + job[-1][1]

    per_machine = {}
    for j, job in enumerate(instance):
        for k, (mid, dur) in enumerate(job):
            per_machine.setdefault(mid, []).append((j,k,dur))

    M = sum(d for job in instance for _, d in job)

    for mid, ops in per_machine.items():
        for (j1, k1, d1), (j2,k2,d2) in itertools.combinations(ops,2):
            z = pulp.LpVariable(f"z_{j1}_{k1}_{j2}_{k2}", cat="Binary")

            prob += s[(j1,k1)] + d1 <= s[(j2,k2)] + M * (1 - z)
            prob += s[(j2,k2)] + d2 <= s[(j1,k1)] + M * z
        


    return prob,s, Cmax

In [22]:
inst = load("data/ft06.txt")
prob, s, Cmax = build(inst)
len(s)
print(len(prob.variables()))
print(len(prob.constraints))

127
216


In [ ]:
def ilp(instance, time_limit=18000, log="results/cplex.log"):
    prob, s, Cmax = build(instance)
    nv= len(prob.variables()) 
    nc = len(prob.constraints)

    t0 = time.perf_counter()
    prob.solve(pulp.CPLEX_CMD(
        path=CPX, msg=0, timeLimit=time_limit, logPath=log,
        keepFiles=False,
        options=["set output clonelog -1"]

    ))
    dt = time.perf_counter() - t0

    makesp  = int(round(Cmax.value()))

    return makesp


In [48]:
print(ilp(inst))

55
